In [1]:
import os
import numpy as np
from util import easyvvuq as uq
import chaospy as cp
import matplotlib.pyplot as plt
from easyvvuq.actions import CreateRunDirectory, Encode, Decode, ExecuteLocal, Actions
from util import persist

/home/mmachado/HPC/energyuq/.venv/lib/python3.12/site-packages/chaospy/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# EasyVVUQ on many Benchmarks

## Setting up an EasyVVUQ campaign

In [2]:
# Quantity of Interest
QOI = 'energy_uj'

# location where the run directories are stored
WORK_DIR = '.results'

We first set up the `params` dictionary, in which we specify the name, type and default value of each input
Also the `vary` dictionary, which holds the `chaospy` distribution of each input

In [3]:
params = {}
vary = {}

# Current machine maximum number of cores
params['N_THREADS'] = {'type': 'integer', 'default': 16}
vary['N_THREADS'] = cp.DiscreteUniform(1, 16)

# Levels of Clock speed, for our current machine:
# 2200000 = 0,
# 2800000 = 1,
# 3300000 = 2
params['CLK'] = {'type': 'integer', 'default': 2}
vary['CLK'] = cp.DiscreteUniform(0, 2)

# params['POWER_CAP'] = {'type': 'integer', 'default': 220.0}  # power cap in watts

d = len(params)

In [4]:
# input file encoder
encoder = uq.encoders.GenericEncoder(template_fname='energy.template', delimiter='$', target_filename='input.csv')

The wrapper writes a CSV file `output.csv` containing the energy, in microjoules, used during the programs execution.

In [5]:
# CSV output file decoder
decoder = uq.decoders.SimpleCSV(target_filename='output.csv', output_columns=[QOI])

In [6]:
# Local execution of the wrapper around benchmarks
execute = ExecuteLocal(f'{os.getcwd()}/energy_wrapper.py')

Now we are combine all actions we want to execute into an `Actions` object.

In [7]:
# actions to be undertaken: make rundirs, encode input files, execute local model ensemble, decode output files
actions = Actions(
    CreateRunDirectory(root=WORK_DIR, flatten=True),
    Encode(encoder),
    execute,
    Decode(decoder)
)

The central object in the UQ analysis is a so-called Campaign. This is created as:

In [8]:
campaign = uq.Campaign(name='energy', params=params, actions=actions, work_dir=WORK_DIR)

We now select the adaptive Stochastic Collocation sampler. Here

* `polynomial_order = 1`: should be interpreted in the sparse context as starting the sampling plan with a level 1 quadrature rule for all inputs.
* `quadrature_rule='C'`: selects the Clenshaw Curtis quadrature rule.
* `sparse=True`: selects the sparse grid.
* `growth=True`: selects an exponential growth rule which makes the Clenshaw Curtis rule nested.
* `dimension_adaptive=True`: selects the dimension-adaptive sampler.

In [9]:
sampler = uq.sampling.SCSampler(vary=vary, polynomial_order=5, quadrature_rule='discrete', midpoint_level1=True, sparse=True)
campaign.set_sampler(sampler)

## Run campaign and adaptation

Run the first ensemble, which consists of just a single sample:

In [10]:
campaign.execute(sequential=True).collate(progress_bar=True)

Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 21859676540

Running NONE with 2 threads

energy_uj: Stdout: 21860062949
NONE
    Stdout
        Running NONE with 2 threads
        OMP_NUM_THREADS=2
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 21869491462

Running NONE with 3 threads

energy_uj: Stdout: 21869953108
NONE
    Stdout
        Running NONE with 3 threads
        OMP_NUM_THREADS=3
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 21879287556

Running NONE with 3 threads

energy_uj: Stdout: 21879661682
NONE
    Stdout
        Running NONE with 3 threads
        OMP_NUM_THREADS=3
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 21888809372

Ru

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:00<00:00, 2080.24it/s]


To analyse the results (and execute the dimension adaptivity), we need an `SCAnalysis` object:

In [11]:
analysis = uq.analysis.SCAnalysis(sampler=sampler, qoi_cols=[QOI])

In [12]:
campaign.apply_analysis(analysis)
data_frame = campaign.get_collation_result()

In [13]:
RESULTS_DIR = "run_results/"
persist.save(analysis, sampler, data_frame, vary, [QOI], RESULTS_DIR)